# Colab Model Station（模型工作站）

專為 Google Colab GPU 環境設計的自動化、可重現推論部署框架。
支援多種推論後端：**大型語言模型（vLLM / Ollama）** 與 **影像擴散模型（FastAPI + Diffusers / ComfyUI）**，具備 **FLUX**、**SDXL** 與 **LoRA** 動態切換能力，可透過 **Tailscale Mesh 私網直連**、**Cloudflare 公網通道** 或 **VS Code Remote Tunnel 遠端開發** 進行連線。

---

## 步驟 0：專案初始化與硬體檢測
將 `colab-inference-node` 複製至執行階段儲存空間（`/content/colab-inference-node`），注入 Colab Secrets 機密，並檢測 GPU 硬體配置與模型型錄。

In [ ]:
# ==============================================================================
# 步驟 0：複製儲存庫、注入機密與檢測硬體
# ==============================================================================
import os
from google.colab import userdata

# 1. 從 Colab Secrets 讀取機密（若已設定）
try:
    os.environ['TAILSCALE_AUTHKEY'] = userdata.get('TAILSCALE_AUTHKEY')
except Exception:
    pass

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

# 2. 複製或更新儲存庫
!git clone https://github.com/yixuanw99/colab-inference-node.git /content/colab-inference-node 2>/dev/null || (cd /content/colab-inference-node && git pull)

# 3. 輸出硬體狀態報告與模型清單
!cd /content/colab-inference-node && bash engine.sh status
!cd /content/colab-inference-node && bash engine.sh list


## 步驟 1：環境依賴安裝

依據目標工作流程安裝對應相依套件：
- **影像擴散引擎** (`bash setup.sh diffusers`)：安裝 FastAPI、Diffusers、PEFT 與 Accelerate。
- **Ollama 語言模型** (`bash setup.sh ollama`)：獨立輕量化 GGUF 執行環境（約 15 秒完成安裝）。
- **vLLM 高並發引擎** (`bash setup.sh vllm`)：具備 PagedAttention 的企業級推論伺服器。
- **ComfyUI 節點圖** (`bash setup.sh comfyui`)：複製 ComfyUI 核心與依賴套件。
- **遠端穿透網路** (`bash setup.sh tunnels`)：安裝 Tailscale、Cloudflared 與 VS Code CLI。
- **完整安裝** (`bash setup.sh all`)：安裝核心基礎工具與依賴。

In [ ]:
%%bash
cd /content/colab-inference-node

# 選擇安裝模式：
# 選項 A：影像擴散模型 (FLUX / SDXL)
bash setup.sh diffusers

# 選項 B：Ollama 語言模型 (適用 T4 / L4)
# bash setup.sh ollama

# 選項 C：vLLM 企業級語言模型 (適用 A100 / L4)
# bash setup.sh vllm


## 硬體規格與模型分配矩陣

| GPU 等級 | 顯存 (VRAM) | 推論引擎 | 目標模型 | 精度 / 格式 | 記憶體卸載策略 | 適用工作負載 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **NVIDIA T4** | 15 GB | **Ollama** | `qwen2.5-coder:7b` | GGUF Q4_K_M | 全顯存常駐 | 輕量化代碼輔助與交談 |
| **NVIDIA T4** | 15 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | 序列循序卸載 (Sequential) | 4 步高速擴散生成 |
| **NVIDIA T4** | 15 GB | **Diffusers** | `stabilityai/stable-diffusion-xl-base-1.0` | FP16 | 模型卸載 (Model Offload) | SDXL 潛在擴散生成 |
| **NVIDIA L4** | 24 GB | **Ollama** | `qwen2.5-coder:32b` | GGUF Q4_K_M | 全顯存常駐 | 高階 IDE 程式開發 |
| **NVIDIA L4** | 24 GB | **vLLM** | `Qwen/Qwen2.5-Coder-14B-Instruct-AWQ` | 4-bit AWQ | 全顯存常駐 | 高並發批次伺服服務 |
| **NVIDIA L4** | 24 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | FP8 | 模型卸載 (Model Offload) | 高速 FLUX 生成 (~15秒) |
| **NVIDIA A100** | 40/80 GB | **vLLM** | `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` | bfloat16 | 全顯存常駐 | 未量化旗艦推理模型 |
| **NVIDIA A100** | 40/80 GB | **Diffusers** | `black-forest-labs/FLUX.1-schnell` | BF16 | 全顯存常駐 | 原生全精度 FLUX (~5秒) |

---

## 部署方案 A：影像擴散服務（FLUX.1 / SDXL + LoRA）經由 Tailscale

於 8000 埠啟動 FastAPI Diffusers 服務並發布至你的私有 Tailnet。

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. 以 Userspace 模式連線至 Tailscale Mesh 網路
bash engine.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. 啟動 FastAPI Diffusers 推論服務 (FLUX.1-schnell FP8 模式)
bash engine.sh diffusers start --model "black-forest-labs/FLUX.1-schnell" --precision fp8

# 3. 將 8000 埠發布至 Tailnet 內部
bash engine.sh tunnel tailscale serve 8000

# 4. 啟動閒置守護行程 (預設 30 分鐘自動釋放運算單元)
bash engine.sh watchdog start --port 8000 --timeout 1800

echo "================================================================"
echo "Diffusers Tailnet 連線端點: http://colab-model-station:8000"
echo "================================================================"


## 部署方案 B：大型語言模型服務（vLLM / Ollama）經由 Tailscale

啟動 OpenAI 相容端點，供本地 OpenCode、Cursor 或 Aider 直連使用。

In [ ]:
%%bash
cd /content/colab-inference-node

# 1. 確保 Tailscale 處於連線狀態
bash engine.sh tunnel tailscale up "$TAILSCALE_AUTHKEY"

# 2. 選擇語言模型推論後端：
# 選項 B1: Ollama (11434 埠)
bash engine.sh ollama start
bash engine.sh ollama pull qwen2.5-coder:7b
bash engine.sh tunnel tailscale serve 11434

# 選項 B2: vLLM (8000 埠, 適用 A100 / L4)
# bash engine.sh vllm start --model "Qwen/Qwen2.5-Coder-14B-Instruct-AWQ"
# bash engine.sh tunnel tailscale serve 8000


## 部署方案 C：VS Code Remote Tunnel 遠端開發（網頁與桌面端）

透過 `https://vscode.dev/tunnel/<name>` 實現零配置遠端開發。若有掛載 Google Drive，認證憑證將自動永久保存。

In [ ]:
%%bash
cd /content/colab-inference-node

# 安裝、登入 GitHub 並啟動 VS Code Tunnel
bash engine.sh tunnel vscode setup
bash engine.sh tunnel vscode login
bash engine.sh tunnel vscode start "colab-model-station"


## 步驟 2：筆記本內推論測試

直接在筆記本中驗證活躍服務的回應狀態。

In [ ]:
import requests
import base64
from IPython.display import display, Image

# 測試擴散生成輸出
try:
    res = requests.post(
        "http://127.0.0.1:8000/v1/images/generations",
        json={"prompt": "A futuristic laboratory on Mars, hyperrealistic", "num_inference_steps": 4, "return_base64": True},
        timeout=60
    ).json()
    print(f"影像擴散生成成功 (耗時: {res['generation_time_sec']}秒)")
    display(Image(base64.b64decode(res['images'][0]['base64_data'])))
except Exception as e:
    print(f"擴散推論狀態提示: {e}")


## 步驟 3：運算單元保護與執行階段釋放

> **重要提醒**：單純關閉瀏覽器分頁**不會**停止 Colab 虛擬機，計費運算單元將持續扣除至伺服器逾時為止。

工作完成後請務必執行此儲存格，立即釋放所有 GPU 顯存並中斷 Colab 執行階段。

In [ ]:
# ==============================================================================
# 步驟 3：終止 Colab 執行階段並停止計費
# ==============================================================================
from google.colab import runtime
print("正在中斷 Google Colab 執行階段以釋放運算單元...")
runtime.unassign()
